# ECG and physiological time-series models: worked solution

Worked solution for the exercise in the lesson **[ECG and physiological time-series models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/ecg-time-series-models)** (*Deep learning & imaging* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](04-ecg-time-series-models.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Physiological signals such as ECGs, oxygen saturation and continuous glucose readings are time series: measurements sampled many times per second or minute. They can be modelled with features designed by experts, such as the variability of the interval between heartbeats, or with neural networks such as 1D CNNs that learn directly from the raw signal. On clean signals both can work well; on noisy signals hand-made features that depend on detecting every beat break down faster. Signal quality, device type and sampling rate shape what any model can do.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Synthetic rhythm strips

Synthetic ECG-like strips with beats, baseline wander and noise. Not real ECGs.

In [ ]:
import numpy as np

FS, SECONDS = 100, 10          # 100 samples per second (real ECGs are often 250-500 Hz), 10-second strips

def synthetic_strip(irregular, rng, noise):
    """A crude single-lead ECG-like strip: a spike per beat, baseline wander and noise.
    irregular=True mimics atrial fibrillation; regular strips include occasional ectopic beats."""
    t = np.arange(FS * SECONDS) / FS
    signal = 0.15 * np.sin(2 * np.pi * 0.3 * t) + rng.normal(0, noise, t.size)
    mean_rr, beat = rng.uniform(0.65, 1.0), rng.uniform(0, 0.5)      # 60-90 beats per minute
    while beat < SECONDS:
        signal += rng.uniform(0.6, 1.2) * np.exp(-((t - beat) ** 2) / (2 * 0.012 ** 2))
        if irregular:
            beat += mean_rr * rng.uniform(0.6, 1.4)
        elif rng.random() < 0.08:
            beat += mean_rr * 0.6                                     # occasional ectopic beat
        else:
            beat += mean_rr * rng.normal(1, 0.03)
    return signal

def make_dataset(n, noise, seed=0):
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 2, n)                                    # 1 = irregular rhythm
    return np.stack([synthetic_strip(bool(l), rng, noise) for l in labels]), labels

signals, labels = make_dataset(2000, noise=0.08)
print(f"{signals.shape[0]} strips of {signals.shape[1]} samples ({SECONDS} s at {FS} Hz); "
      f"{labels.mean():.0%} irregular")

# 2000 strips of 1000 samples (10 s at 100 Hz); 52% irregular

### Approach 1: expert features

Continuing from the previous block: detect beats, compute RR-interval variability, fit logistic regression.

In [ ]:
from scipy.signal import find_peaks
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def rr_features(signal):
    peaks, _ = find_peaks(signal, height=0.4, distance=int(0.3 * FS))   # detect beats
    rr = np.diff(peaks) / FS
    return [np.std(rr) / np.mean(rr), len(peaks)]                      # RR variability, beat count

def feature_model_auroc(signals, labels):
    feats = np.array([rr_features(s) for s in signals])
    tr, te = train_test_split(np.arange(len(labels)), test_size=0.3, random_state=0, stratify=labels)
    p = LogisticRegression().fit(feats[tr], labels[tr]).predict_proba(feats[te])[:, 1]
    return roc_auc_score(labels[te], p)

print(f"RR-variability features + logistic regression: AUROC {feature_model_auroc(signals, labels):.3f}")

# RR-variability features + logistic regression: AUROC 0.917

### Approach 2: a 1D CNN on the raw signal

Continuing from the previous blocks: a three-layer 1D CNN that learns from the raw samples.

In [ ]:
import torch
from torch import nn

def cnn_auroc(signals, labels, epochs=15):
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Conv1d(1, 8, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(4),
        nn.Conv1d(8, 16, kernel_size=7, padding=3), nn.ReLU(), nn.MaxPool1d(4),
        nn.Conv1d(16, 16, kernel_size=7, padding=3), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        nn.Flatten(), nn.Linear(16, 2))
    X, y = torch.tensor(signals, dtype=torch.float32).unsqueeze(1), torch.tensor(labels)
    tr, te = train_test_split(np.arange(len(labels)), test_size=0.3, random_state=0, stratify=labels)
    tr = torch.tensor(tr)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(epochs):
        for batch in tr[torch.randperm(len(tr))].split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[batch]), y[batch]).backward()
            opt.step()
    with torch.no_grad():
        p = model(X[te]).softmax(1)[:, 1].numpy()
    return roc_auc_score(labels[te], p)

print(f"1D CNN on the raw signal:                      AUROC {cnn_auroc(signals, labels):.3f}")

# 1D CNN on the raw signal:                      AUROC 0.945

### When the signal is noisy

Continuing from the previous blocks: the same two approaches on much noisier strips.

In [ ]:
noisy_signals, noisy_labels = make_dataset(2000, noise=0.25, seed=1)
print(f"Noisy strips, RR features: AUROC {feature_model_auroc(noisy_signals, noisy_labels):.3f}")
print(f"Noisy strips, 1D CNN:      AUROC {cnn_auroc(noisy_signals, noisy_labels):.3f}")

# Noisy strips, RR features: AUROC 0.601
# Noisy strips, 1D CNN:      AUROC 0.812

## Exercise

**Goal:** Compare feature-based and deep-learning rhythm classifiers on synthetic and real ECGs.

**Data:** Synthetic strips in the notebook; PTB-XL 12-lead ECG dataset from PhysioNet (optional) (Generated in the notebook; PTB-XL under CC BY 4.0)

### Step 1

Reproduce both approaches on clean and noisy strips, and plot an example of each class.

In [ ]:
import matplotlib.pyplot as plt

print(f"Clean: features {feature_model_auroc(signals, labels):.3f}, CNN {cnn_auroc(signals, labels):.3f}")
print(f"Noisy: features {feature_model_auroc(noisy_signals, noisy_labels):.3f}, CNN {cnn_auroc(noisy_signals, noisy_labels):.3f}")

t = np.arange(FS * SECONDS) / FS
fig, axes = plt.subplots(2, 2, figsize=(11, 4), sharex=True)
for row, (sig, lab, name) in enumerate([(signals, labels, "clean"), (noisy_signals, noisy_labels, "noisy")]):
    for col, cls in enumerate([0, 1]):
        i = np.where(lab == cls)[0][0]
        axes[row, col].plot(t, sig[i], linewidth=0.7)
        axes[row, col].set_title(f"{name}, {'irregular' if cls else 'regular'}", fontsize=9)
axes[1, 0].set_xlabel("seconds"); axes[1, 1].set_xlabel("seconds")
plt.tight_layout(); plt.show()

### Step 2

Add a signal-quality score (for example, the fraction of detected beats with plausible RR intervals) and exclude low-quality strips; report AUROC and the share excluded.

In [ ]:
from scipy.stats import kurtosis

def quality_score(signal):
    """Kurtosis of the strip: clean ECG is dominated by sharp QRS spikes (high kurtosis); noise lowers it.
    Kurtosis-based signal-quality indices are a standard first check in ECG processing."""
    return float(kurtosis(signal))

q = np.array([quality_score(s) for s in noisy_signals])
print(f"Quality score: clean strips median {np.median([quality_score(s) for s in signals]):.1f}, "
      f"noisy strips median {np.median(q):.1f}")
for exclude in [0.0, 0.25, 0.5]:
    threshold = np.quantile(q, exclude)
    keep = q >= threshold
    excluded_irregular = noisy_labels[~keep].mean() if (~keep).any() else float("nan")
    print(f"exclude lowest {exclude:.0%} by quality: features AUROC {feature_model_auroc(noisy_signals[keep], noisy_labels[keep]):.3f}; "
          f"irregular share among excluded {excluded_irregular:.0%} (overall {noisy_labels.mean():.0%})")

The kurtosis score separates clean from noisy strips clearly (medians 7.3 against 1.2), but excluding the
lowest-quality noisy strips does not improve AUROC here. In this simulation every noisy strip has the same noise
level, so the "worst" strips are not really worse; excluding them just shrinks the sample. In real recordings noise
varies from strip to strip, and quality filtering usually helps on the strips kept. Either way, report the share
excluded, and check whether exclusions fall unevenly on one class (slightly more irregular rhythms among the excluded
here, 54% against 51% overall), because a quality filter can quietly remove the cases you most want to find.

### Step 3

Add a third feature, the proportion of successive RR differences above 50 ms, and see whether the feature model improves.

In [ ]:
def rr_features_3(signal):
    peaks, _ = find_peaks(signal, height=0.4, distance=int(0.3 * FS))
    rr = np.diff(peaks) / FS
    if len(rr) < 3:
        return [0.0, len(peaks), 0.0]
    pnn50 = np.mean(np.abs(np.diff(rr)) > 0.05)          # share of successive RR differences above 50 ms
    return [np.std(rr) / np.mean(rr), len(peaks), pnn50]

def auroc_with(feature_fn, sigs, labs):
    feats = np.array([feature_fn(s) for s in sigs])
    tr, te = train_test_split(np.arange(len(labs)), test_size=0.3, random_state=0, stratify=labs)
    p = LogisticRegression(max_iter=1000).fit(feats[tr], labs[tr]).predict_proba(feats[te])[:, 1]
    return roc_auc_score(labs[te], p)

for name, sigs, labs in [("clean", signals, labels), ("noisy", noisy_signals, noisy_labels)]:
    print(f"{name}: two features {auroc_with(rr_features, sigs, labs):.3f}, "
          f"with pNN50 {auroc_with(rr_features_3, sigs, labs):.3f}")

pNN50 measures beat-to-beat irregularity directly, so it adds information that the overall coefficient of
variation partly misses (a regular rhythm with occasional ectopic beats can have a high coefficient of variation
but few large successive differences). At 100 samples per second, 50 ms is only five samples, so real ECG work
uses higher sampling rates.

### Step 4

Optional: train the 1D CNN on PTB-XL lead II to detect atrial fibrillation, splitting by patient as the dataset recommends.

Optional, outline (not run here because PTB-XL is a large download). PTB-XL is an openly available dataset of
21,837 12-lead ECGs from 18,885 patients on PhysioNet (Wagner et al., Scientific Data 2020), provided at 100 Hz and
500 Hz with diagnostic statements. Check its licence on PhysioNet before use.

```python
import ast, numpy as np, pandas as pd, wfdb          # pip install wfdb
root = "ptb-xl/"                                      # after downloading the dataset from PhysioNet
meta = pd.read_csv(root + "ptbxl_database.csv", index_col="ecg_id")
meta["codes"] = meta.scp_codes.apply(ast.literal_eval)
meta["af"] = meta.codes.apply(lambda c: int("AFIB" in c))
lead_ii = lambda f: wfdb.rdsamp(root + f)[0][:, 1]   # 100 Hz records: filename_lr; column 1 is lead II
X = np.stack([lead_ii(f) for f in meta.filename_lr])
# The dataset assigns every record to one of ten folds, keeping all of a patient's records in one fold.
# The recommended split: folds 1-8 for training, 9 for validation, 10 for testing.
train, val, test = meta.strat_fold <= 8, meta.strat_fold == 9, meta.strat_fold == 10
```

Then train the same 1D CNN on `X[train]` (1,000 samples per record at 100 Hz, so the architecture fits directly),
tune on `val`, and report AUROC with a confidence interval on `test` only once. Splitting by fold rather than at
random keeps the same patient out of both training and test data.

---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/ecg-time-series-models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/ecg-time-series-models). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.